In [1]:
#%pip install -q --upgrade openai pandas tqdm

In [2]:
from __future__ import annotations

import hashlib
import json
import os
import re
import time

from concurrent.futures import (ThreadPoolExecutor, as_completed)
from datetime import (datetime, timezone)
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from openai import OpenAI
from tqdm.auto import tqdm

In [3]:
BASE_URL = ("http://api.llm.apps.os.dcs.gla.ac.uk/v1")

MODEL_NAME = "gpt-oss-120b"

if "IDA_LLM_API_KEY" not in os.environ:
    raise EnvironmentError("IDA_LLM_API_KEY is not set in the environment.")

client = OpenAI(
    base_url=BASE_URL,
    api_key=os.environ["IDA_LLM_API_KEY"],
    timeout=180.0,
)


# ============================================================
# INPUT DATA
# ============================================================


SNAPSHOT_ROOT = Path(
    "/mnt/primary/Finnhub Pipeline/finnhub_snapshots_filtered"
)

ANSWER_ROOT = Path("/mnt/primary/Finnhub Pipeline/finnhub_answers")

START_DATE = pd.Timestamp("2026-07-15")

END_DATE = pd.Timestamp("2026-08-28")

OUTPUT_ROOT = Path("/mnt/primary/Finnhub Pipeline/llm_financial_features")

DAILY_JSON_ROOT = (OUTPUT_ROOT / "daily_json")

MASTER_CSV_PATH = (OUTPUT_ROOT / "llm_financial_features.csv")

PROCESSING_LOG_PATH = (OUTPUT_ROOT / "processing_log.csv")

COVERAGE_REPORT_PATH = (OUTPUT_ROOT / "daily_coverage_report.csv")

INVALID_JSON_REPORT_PATH = (OUTPUT_ROOT / "invalid_output_files.csv")

MAX_WORKERS = 10

MAX_API_ATTEMPTS = 4
RETRY_BASE_SECONDS = 2.0

MAX_OUTPUT_TOKENS = 1000

NEWS_MAX_CHARS = 50_000
ANSWER_MAX_CHARS = 120_000
PREVIOUS_ANSWER_MAX_CHARS = 60_000

PROMPT_VERSION = ("financial_features")

PROCESSOR_VERSION = ("llm_feature_extractor_v1")

FORCE_REPROCESS = False

TEST_MODE = False
TEST_LIMIT = 10


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DAILY_JSON_ROOT.mkdir(parents=True, exist_ok=True,)

print("Model:", MODEL_NAME)
print("Filtered snapshots:", SNAPSHOT_ROOT)
print("Answers:", ANSWER_ROOT)
print("Date range:", START_DATE.date(), "to", END_DATE.date())
print("Output:", OUTPUT_ROOT)

Model: gpt-oss-120b
Filtered snapshots: /mnt/primary/Finnhub Pipeline/finnhub_snapshots_filtered
Answers: /mnt/primary/Finnhub Pipeline/finnhub_answers
Date range: 2026-07-15 to 2026-08-28
Output: /mnt/primary/Finnhub Pipeline/llm_financial_features


In [4]:
EVENT_TYPES = [
    "earnings",
    "guidance",
    "analyst_rating",
    "product",
    "partnership",
    "merger_acquisition",
    "regulation",
    "litigation",
    "management",
    "capital_return",
    "financing",
    "operations",
    "macro_exposure",
    "other",
    "none",
]

print(EVENT_TYPES)

['earnings', 'guidance', 'analyst_rating', 'product', 'partnership', 'merger_acquisition', 'regulation', 'litigation', 'management', 'capital_return', 'financing', 'operations', 'macro_exposure', 'other', 'none']


In [5]:
def normalise_whitespace(text: str) -> str:
    text = str(text).replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_date_from_folder(path: Path) -> pd.Timestamp | None:
    try:
        return pd.Timestamp(path.name).normalize()
    except Exception:
        return None


def safe_filename(value: str) -> str:
    value = re.sub(r'[<>:"/\\|?*]', "_", str(value))
    value = re.sub(r"\s+", " ", value).strip()
    return value


def normalise_company_key(value: str) -> str:
    value = str(value).lower()

    value = value.replace("&", "and")

    value = re.sub(r"_answer_\d{4}-\d{2}-\d{2}$", "", value, flags=re.IGNORECASE)

    value = re.sub(r"[^a-z0-9]+", "", value)

    return value


def sha256_file(path: Path | None, chunk_size: int = 1024 * 1024) -> str | None:
    if path is None:
        return None

    path = Path(path)

    if not path.exists():
        return None

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def truncate_text(text: str, max_chars: int) -> tuple[str, bool]:

    text = str(text)

    if len(text) <= max_chars:
        return text, False

    first_chars = int(max_chars * 0.75)

    last_chars = (max_chars - first_chars)

    truncated = (text[:first_chars] + "\n\n[... INPUT TRUNCATED ...]\n\n" + text[-last_chars:])

    return truncated, True


def clamp(value: Any, lower: float, upper: float) -> float:
    number = float(value)
    return float(min(max(number, lower), upper))

In [6]:
def load_snapshot_file(snapshot_path: Path) -> dict[str, Any]:

    with open(snapshot_path, "r", encoding="utf-8") as file:
        raw = json.load(file)

    snapshot_date = pd.to_datetime(raw.get("snapshot_date"), errors="coerce")

    if pd.isna(snapshot_date):
        snapshot_date = (parse_date_from_folder(snapshot_path.parent))

    ticker = normalise_whitespace(raw.get("ticker", "")).upper()

    company_name = normalise_whitespace(raw.get("company_name", ""))

    profile = (raw.get("data", {}).get("company_profile", {}) or {})

    if not ticker:
        ticker = normalise_whitespace(
            profile.get("ticker", "")).upper()

    if not company_name:
        company_name = normalise_whitespace(profile.get("name", ""))

    if (snapshot_date is None or pd.isna(snapshot_date)):
        raise ValueError(f"No valid date in {snapshot_path}")

    if not ticker:
        raise ValueError(f"No ticker in {snapshot_path}")

    if not company_name:
        company_name = ticker

    return {
        "snapshot_date": (pd.Timestamp(snapshot_date).normalize()),
        "ticker": ticker,
        "company_name": (company_name),
        "category": raw.get("category"),
        "data": raw.get("data", {}),
        "source_path": (str(snapshot_path))}

In [7]:
def extract_filtered_news(snapshot: dict[str, Any]) -> list[dict[str, Any]]:

    news_items = (snapshot.get("data",{}).get("company_news", []) or [])

    extracted = []
    seen = set()

    for item in news_items:

        if not isinstance(item, dict):
            continue
            
        headline = (normalise_whitespace(item.get("headline", "")))

        summary = (normalise_whitespace(item.get("summary", "")))

        if not headline:
            continue

        dedupe_key = (headline.lower())

        if dedupe_key in seen:
            continue

        seen.add(dedupe_key)

        extracted.append({
                "headline": headline,
                "summary": summary,
                "source": item.get("source"),
                "datetime_utc": (item.get("datetime_utc"))
            }
        )

    return extracted


def format_news_for_prompt(news_items: list[dict[str, Any]]) -> str:

    sections = []

    for index, item in enumerate(news_items,start=1):
        section = (
            f"ARTICLE {index}\n"
            f"Headline: {item['headline']}\n"
        )

        if item.get("summary"):
            section += (
                f"Summary: "
                f"{item['summary']}\n"
            )

        if item.get("source"):
            section += (f"Source: "f"{item['source']}\n")

        if item.get("datetime_utc"):
            section += (f"Time: "f"{item['datetime_utc']}\n")
        sections.append(section.strip())

    return "\n\n".join(sections)

In [8]:
def build_answer_index(answer_root: Path, start_date: pd.Timestamp, end_date: pd.Timestamp) -> dict[tuple[pd.Timestamp, str], Path]:

    answer_index = {}

    for date_directory in sorted(answer_root.iterdir()):
        if not date_directory.is_dir():
            continue

        folder_date = (parse_date_from_folder(date_directory))

        if folder_date is None:
            continue

        if folder_date < start_date:
            continue

        if folder_date > end_date:
            continue

        for answer_path in (date_directory.glob("*.txt")):
            company_part = re.sub(r"_answer_\d{4}-\d{2}-\d{2}$", "", answer_path.stem, flags=re.IGNORECASE)
            company_key = (normalise_company_key(company_part))
            answer_index[(folder_date, company_key)] = answer_path

    return answer_index


def find_answer_path(
    snapshot_date: pd.Timestamp,
    company_name: str,
    ticker: str,
    answer_index: dict[tuple[pd.Timestamp, str], Path]
) -> Path | None:

    candidate_keys = [
        normalise_company_key(company_name),
        normalise_company_key(company_name.replace(".", "")),
        normalise_company_key(ticker)
    ]

    for company_key in (candidate_keys):
        path = answer_index.get((snapshot_date, company_key))
        if path is not None:
            return path


    target_key = (normalise_company_key(company_name))

    same_date = [
        (key, path)
        for (date, key), path in (answer_index.items())
        if date == snapshot_date
    ]

    for (candidate_key, path) in same_date:
        if (target_key in candidate_key or candidate_key in target_key):
            return path
    return None


def find_previous_answer_path(
    snapshot_date: pd.Timestamp,
    company_name: str,
    ticker: str,
    answer_index: dict[tuple[pd.Timestamp, str], Path]
) -> Path | None:


    candidate_keys = {
        normalise_company_key(company_name),
        normalise_company_key(company_name.replace(".", "")),
        normalise_company_key(ticker)
    }

    candidates = []

    for (date, key), path in answer_index.items():

        if date >= snapshot_date:
            continue

        direct_match = (key in candidate_keys)

        fuzzy_match = any(
            candidate
            and (candidate in key or key in candidate) for candidate in candidate_keys)

        if direct_match or fuzzy_match:
            candidates.append((date, path))

    if not candidates:
        return None

    candidates.sort(key=lambda item: item[0], reverse=True)

    return candidates[0][1]


def load_answer_text(answer_path: Path) -> str:

    text = answer_path.read_text(encoding="utf-8", errors="replace")

    lines = []

    for raw_line in (text.splitlines()):
        line = (raw_line.strip())

        if not line:
            continue

        if re.fullmatch(r"[-=_*]{3,}", line):
            continue

        lines.append(line)

    return "\\n".join(lines)

In [9]:
NEWS_SYSTEM_PROMPT = """
You are a financial feature-extraction engine for an academic stock-forecasting experiment.

Use ONLY the company-specific news text supplied by the user.
Do not use outside knowledge, future events, market prices, or information not explicitly contained in the supplied text.
Do not attempt to predict an exact percentage return.

Return exactly one valid JSON object and no markdown or commentary.
""".strip()


def build_news_prompt(
    company_name: str,
    ticker: str,
    feature_date: pd.Timestamp,
    news_text: str,
) -> str:

    event_types_text = ", ".join(EVENT_TYPES)

    return f"""
COMPANY: {company_name}
TICKER: {ticker}
FEATURE DATE: {feature_date.date()}

You are assessing ONLY the filtered news items below.

Extract these features:

1. sentiment
   Range: -1 to +1.
   -1 = strongly negative textual/financial tone.
    0 = neutral or balanced.
   +1 = strongly positive textual/financial tone.

2. impact_1d
3. impact_3d
4. impact_5d
   Each range: -1 to +1.
   These represent the expected direction and strength of stock-price pressure
   caused by the supplied information over 1, 3, and 5 TRADING DAYS.
   They are NOT predicted percentage returns.
   Use 0 when no meaningful directional impact is supported.

5. confidence
   Range: 0 to 1.
   Confidence that the supplied news supports the impact assessment.
   If evidence is weak, contradictory, vague, or mostly commentary, lower this score.

6. novelty
   Range: 0 to 1.
   0 = routine, repetitive, recycled, or little genuinely new information.
   1 = clearly new and distinctive information.
   Judge only from the supplied items; do not assume knowledge of prior news.

7. materiality
   Range: 0 to 1.
   0 = financially immaterial.
   1 = potentially very important for earnings, cash flow, valuation,
       operations, competitive position, or investor expectations.

8. event_type
   Choose exactly ONE dominant category from:
   {event_types_text}

Important rules:
- Use only supplied text.
- A positive-sounding headline does not necessarily imply positive market impact.
- Consider expectation effects when the supplied text explicitly supports them.
- If stories conflict, reflect that in lower confidence and more neutral impact.
- Do not infer facts not present in the text.

Return EXACTLY:

{{
  "sentiment": 0.0,
  "impact_1d": 0.0,
  "impact_3d": 0.0,
  "impact_5d": 0.0,
  "confidence": 0.0,
  "novelty": 0.0,
  "materiality": 0.0,
  "event_type": "other"
}}

NEWS:
{news_text}
""".strip()

In [10]:
ANSWER_SYSTEM_PROMPT = """
You are a financial feature-extraction engine for an academic stock-forecasting experiment.

Use ONLY the supplied answer text.
Do not use outside knowledge, future events, market prices, or information not explicitly contained in the supplied text.
Do not attempt to predict an exact percentage return.

Return exactly one valid JSON object and no markdown or commentary.
""".strip()


def build_answer_prompt(
    company_name: str,
    ticker: str,
    feature_date: pd.Timestamp,
    answer_text: str,
    previous_answer_text: str | None,
) -> str:

    previous_block = (
        previous_answer_text
        if previous_answer_text
        else "NONE - no previous answer file is available for comparison."
    )

    return f"""
COMPANY: {company_name}
TICKER: {ticker}
FEATURE DATE: {feature_date.date()}

The CURRENT ANSWER text below consists of analytical answers generated from
financial/company data available for this company on the feature date.

A PREVIOUS ANSWER may also be supplied. It is provided ONLY so that you can
measure how much the current answer has materially changed.

Extract these features from the CURRENT ANSWER:

1. impact_1d
2. impact_3d
3. impact_5d
   Each range: -1 to +1.
   These represent the expected direction and strength of stock-price pressure
   supported by the CURRENT answers over 1, 3, and 5 TRADING DAYS.
   They are NOT predicted percentage returns.
   Use 0 when the current answers do not support a meaningful directional impact.

4. confidence
   Range: 0 to 1.
   Confidence that the CURRENT answers support the impact assessment.
   Lower this if the answers repeatedly state that information is missing,
   inconclusive, contradictory, or unavailable.

5. fundamental_strength
   Range: -1 to +1.
   -1 = very weak fundamental condition.
    0 = neutral/mixed/insufficient evidence.
   +1 = very strong fundamental condition.
   Base this only on evidence contained in the CURRENT answers.

6. catalyst_score
   Range: -1 to +1.
   -1 = strong negative catalysts.
    0 = no meaningful catalysts / balanced.
   +1 = strong positive catalysts.

7. risk_score
   Range: 0 to 1.
   0 = little material downside risk identified.
   1 = severe downside, uncertainty, or adverse risk identified.

8. information_quality
   Range: 0 to 1.
   0 = almost no useful, specific evidence is available.
   1 = strong, specific, internally useful evidence supports the assessment.

9. novelty
   Range: 0 to 1.
   Compare the CURRENT ANSWER with the PREVIOUS ANSWER.
   0.0 = essentially the same evidence, conclusions, risks, and catalysts.
   0.5 = some meaningful changes but the overall picture is similar.
   1.0 = materially new evidence, changed conclusions, or a substantially
         different risk/catalyst/fundamental picture.

   IMPORTANT:
   - Novelty measures CHANGE, not positivity or negativity.
   - Do not give a high novelty score just because the current answer is detailed.
   - Rewording the same information is NOT novel.
   - If PREVIOUS ANSWER is NONE, return novelty = 0.0 because change cannot
     be measured yet.

Important rules:
- Use only supplied answer text.
- Do not infer missing facts.
- Do not use outside knowledge.
- If evidence is missing, keep impacts near 0 and confidence/information_quality low.
- Distinguish long-term fundamental strength from short-horizon market impact.
- Use PREVIOUS ANSWER only for novelty comparison. All other feature scores
  must describe the CURRENT ANSWER.

Return EXACTLY:

{{
  "impact_1d": 0.0,
  "impact_3d": 0.0,
  "impact_5d": 0.0,
  "confidence": 0.0,
  "fundamental_strength": 0.0,
  "catalyst_score": 0.0,
  "risk_score": 0.0,
  "information_quality": 0.0,
  "novelty": 0.0
}}

CURRENT ANSWER:
{answer_text}

PREVIOUS ANSWER FOR NOVELTY COMPARISON:
{previous_block}
""".strip()

In [11]:
def extract_json_object(raw_text: str) -> dict[str, Any]:

    text = str(raw_text).strip()

    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)

    text = re.sub(r"\s*```$", "", text).strip()

    try:
        parsed = json.loads(text)

        if not isinstance(parsed, dict):
            raise ValueError("Response JSON is not an object.")

        return parsed

    except Exception:
        pass

    first = text.find("{")

    last = text.rfind("}")

    if (first == -1 or last == -1 or last <= first):
        raise ValueError( "No JSON object found in model response.")

    parsed = json.loads(text[first:last + 1])

    if not isinstance(parsed, dict):
        raise ValueError("Extracted JSON is not an object.")

    return parsed


def validate_news_features(raw: dict[str, Any]) -> dict[str, Any]:
    required = [
        "sentiment",
        "impact_1d",
        "impact_3d",
        "impact_5d",
        "confidence",
        "novelty",
        "materiality",
        "event_type",
    ]

    missing = [key for key in required if key not in raw]

    if missing:
        raise ValueError(f"Missing news keys: {missing}")

    event_type = str(raw["event_type"]).strip().lower()

    if event_type not in (EVENT_TYPES):
        event_type = "other"

    return {
        "available": True,
        "sentiment": clamp(raw["sentiment"], -1, 1),
        "impact_1d": clamp(raw["impact_1d"], -1, 1),
        "impact_3d": clamp(raw["impact_3d"], -1, 1),
        "impact_5d": clamp(raw["impact_5d"], -1, 1),
        "confidence": clamp(raw["confidence"], 0, 1),
        "novelty": clamp(raw["novelty"], 0, 1),
        "materiality": clamp(raw["materiality"], 0, 1),
        "event_type": event_type
    }


def validate_answer_features(raw: dict[str, Any]) -> dict[str, Any]:

    required = [
        "impact_1d",
        "impact_3d",
        "impact_5d",
        "confidence",
        "fundamental_strength",
        "catalyst_score",
        "risk_score",
        "information_quality",
        "novelty",
    ]

    missing = [key for key in required if key not in raw]

    if missing:
        raise ValueError(f"Missing answer keys: {missing}")

    return {
        "available": True,
        "impact_1d": clamp(raw["impact_1d"], -1, 1),
        "impact_3d": clamp(raw["impact_3d"], -1, 1),
        "impact_5d": clamp(raw["impact_5d"], -1, 1),
        "confidence": clamp(raw["confidence"], 0, 1),
        "fundamental_strength": clamp(raw["fundamental_strength"], -1, 1),
        "catalyst_score": clamp(raw["catalyst_score"], -1, 1),
        "risk_score": clamp(raw["risk_score"], 0, 1),
        "information_quality": clamp(raw["information_quality"], 0, 1),
        "novelty": clamp(raw["novelty"], 0, 1)}

In [12]:
def empty_news_features() -> dict[str, Any]:
    return {
        "available": False,
        "sentiment": 0.0,
        "impact_1d": 0.0,
        "impact_3d": 0.0,
        "impact_5d": 0.0,
        "confidence": 0.0,
        "novelty": 0.0,
        "materiality": 0.0,
        "event_type": "none",
    }


def empty_answer_features() -> dict[str, Any]:
    return {
        "available": False,
        "impact_1d": 0.0,
        "impact_3d": 0.0,
        "impact_5d": 0.0,
        "confidence": 0.0,
        "fundamental_strength": 0.0,
        "catalyst_score": 0.0,
        "risk_score": 0.0,
        "information_quality": 0.0,
        "novelty": 0.0,
    }

In [13]:
def ask_llm_json(system_prompt: str, user_prompt: str, validator) -> tuple[dict[str, Any], str]:

    last_error = None
    last_raw = None

    for attempt in range(1, MAX_API_ATTEMPTS + 1):
        try:
            result = (
                client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    temperature=0.0,
                    max_tokens=MAX_OUTPUT_TOKENS
                )
            )

            raw_response = (result.choices[0].message.content)

            last_raw = (raw_response)

            parsed = (extract_json_object(raw_response))

            validated = validator(parsed)

            return (validated, raw_response)

        except Exception as error:
            last_error = error

            if attempt < MAX_API_ATTEMPTS:
                sleep_seconds = (RETRY_BASE_SECONDS * (2 ** (attempt - 1)))

                time.sleep(sleep_seconds)

    raise RuntimeError(
        "LLM request failed after "
        f"{MAX_API_ATTEMPTS} attempts. "
        f"Last error: {type(last_error).__name__}: "
        f"{last_error}. "
        f"Last raw response: {last_raw!r}"
    )

In [14]:
def discover_snapshot_files(
    snapshot_root: Path,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp
) -> list[Path]:

    if not snapshot_root.exists():
        raise FileNotFoundError(f"Snapshot root not found: "f"{snapshot_root}")

    files = []

    for date_directory in sorted(snapshot_root.iterdir()):
        if not date_directory.is_dir():
            continue

        folder_date = (parse_date_from_folder( date_directory))

        if folder_date is None:
            continue

        if folder_date < start_date:
            continue

        if folder_date > end_date:
            continue

        files.extend(sorted(date_directory.rglob("*.json")))

    return files


snapshot_files = (
    discover_snapshot_files(
        snapshot_root=SNAPSHOT_ROOT,
        start_date=START_DATE,
        end_date=END_DATE
    )
)

if TEST_MODE:
    snapshot_files = (snapshot_files[:TEST_LIMIT])

print("Snapshots discovered:", len(snapshot_files))

if snapshot_files:
    print("First:", snapshot_files[0])
    print("Last:", snapshot_files[-1])

Snapshots discovered: 4401
First: /mnt/primary/Finnhub Pipeline/finnhub_snapshots_filtered/2026-07-15/.ipynb_checkpoints/3M_MMM_snapshot-checkpoint.json
Last: /mnt/primary/Finnhub Pipeline/finnhub_snapshots_filtered/2026-08-28/XYL_snapshot.json


In [15]:
if not ANSWER_ROOT.exists():
    raise FileNotFoundError(
        f"Answer root not found: "
        f"{ANSWER_ROOT}"
    )

answer_index = (
    build_answer_index(
        answer_root=ANSWER_ROOT,
        start_date=START_DATE,
        end_date=END_DATE
    )
)

print("Answer files indexed:", len(answer_index))

Answer files indexed: 4400


In [16]:
def get_output_path(feature_date: pd.Timestamp, ticker: str) -> Path:

    date_directory = (DAILY_JSON_ROOT / str(feature_date.date()))

    date_directory.mkdir(parents=True, exist_ok=True)

    return (date_directory / f"{safe_filename(ticker)}_features.json")


def load_existing_output(output_path: Path) -> dict[str, Any] | None:

    if not output_path.exists():
        return None

    try:
        with open(output_path, "r", encoding="utf-8") as file:
            return json.load(file)
    except Exception:
        return None


def needs_processing(
    output_path: Path,
    snapshot_hash: str | None,
    answer_hash: str | None,
    previous_answer_hash: str | None,
) -> tuple[bool, str]:

    if FORCE_REPROCESS:
        return True, "forced"

    existing = (load_existing_output(output_path))

    if existing is None:
        return True, "missing_or_invalid_output"

    if (existing.get("model") != MODEL_NAME):
        return True, "model_changed"

    if (existing.get( "prompt_version") != PROMPT_VERSION):
        return True, "prompt_changed"

    inputs = existing.get("input_files", {})

    if (inputs.get( "snapshot_sha256") != snapshot_hash):
        return True, "snapshot_changed"

    if (inputs.get("answer_sha256") != answer_hash):
        return True, "answer_changed"

    if (inputs.get("previous_answer_sha256") != previous_answer_hash):
        return True, "previous_answer_changed"

    return False, "already_complete"


def write_json_atomically(output_path: Path, payload: dict[str, Any]) -> None:

    output_path.parent.mkdir(parents=True, exist_ok=True)

    temporary = (output_path.with_suffix(output_path.suffix + ".tmp"))

    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )

    os.replace(temporary, output_path)

In [17]:
def process_one_company_date(snapshot_path: Path) -> dict[str, Any]:

    snapshot = (load_snapshot_file(snapshot_path))

    feature_date = (snapshot["snapshot_date"])

    ticker = (snapshot["ticker"])

    company_name = (snapshot["company_name"])

    if (feature_date < START_DATE or feature_date > END_DATE):
        return {
            "Date": feature_date,
            "Symbol": ticker,
            "Company": company_name,
            "Status": "skipped",
            "Reason": "outside_date_range",
        }

    answer_path = (
        find_answer_path(
            snapshot_date=feature_date,
            company_name=company_name,
            ticker=ticker,
            answer_index=answer_index,
        )
    )

    previous_answer_path = (
        find_previous_answer_path(
            snapshot_date=feature_date,
            company_name=company_name,
            ticker=ticker,
            answer_index=answer_index,
        )
    )

    output_path = (get_output_path(feature_date = feature_date, ticker=ticker))

    snapshot_hash = (sha256_file(snapshot_path))

    answer_hash = (sha256_file(answer_path))

    previous_answer_hash = (sha256_file(previous_answer_path))

    should_process, reason = (
        needs_processing(
            output_path=output_path,
            snapshot_hash=snapshot_hash,
            answer_hash=answer_hash,
            previous_answer_hash=previous_answer_hash
        )
    )

    if not should_process:
        return {
            "Date": feature_date,
            "Symbol": ticker,
            "Company": company_name,
            "Status": "skipped",
            "Reason": reason,
            "Output_Path": str(output_path)
        }

    # ========================================================
    # NEWS
    # ========================================================

    news_items = (extract_filtered_news(snapshot))

    news_raw_response = None
    news_input_truncated = False

    if news_items:
        news_text = (format_news_for_prompt(news_items))

        (news_text, news_input_truncated) = truncate_text(news_text, NEWS_MAX_CHARS)

        news_prompt = (
            build_news_prompt(
                company_name=company_name,
                ticker=ticker,
                feature_date=feature_date,
                news_text=news_text,
            )
        )

        (news_features, news_raw_response) = ask_llm_json(
            system_prompt=(NEWS_SYSTEM_PROMPT),
            user_prompt=(news_prompt),
            validator=(validate_news_features)
        )

    else:
        news_features = (empty_news_features())

    news_features["article_count"] = len(news_items)

    # ========================================================
    # ANSWERS
    # ========================================================

    answer_raw_response = None
    answer_input_truncated = False
    previous_answer_input_truncated = False

    if (answer_path is not None and answer_path.exists()):

        answer_text = (load_answer_text(answer_path))

        if answer_text.strip():

            (answer_text, answer_input_truncated) = truncate_text(answer_text, ANSWER_MAX_CHARS)

            previous_answer_text = None

            if (previous_answer_path is not None and previous_answer_path.exists()):
                
                previous_answer_text = (load_answer_text(previous_answer_path))

                if previous_answer_text.strip():
                    (previous_answer_text, previous_answer_input_truncated) = truncate_text(previous_answer_text, PREVIOUS_ANSWER_MAX_CHARS)
                else:
                    previous_answer_text = None

            answer_prompt = (
                build_answer_prompt(
                    company_name=company_name,
                    ticker=ticker,
                    feature_date=feature_date,
                    answer_text=answer_text,
                    previous_answer_text=previous_answer_text,
                )
            )

            (answer_features, answer_raw_response) = ask_llm_json(
                system_prompt=(ANSWER_SYSTEM_PROMPT),
                user_prompt=(answer_prompt),
                validator=(validate_answer_features)
            )

        else:
            answer_features = (empty_answer_features())

    else:
        answer_features = (empty_answer_features())

    answer_features["previous_available"] = bool(
        previous_answer_path is not None and previous_answer_path.exists())


    news_features["sentiment_signal"] = (news_features["sentiment"] * news_features["confidence"])

    for horizon in ["1d", "3d", "5d"]:

        impact_key = (f"impact_{horizon}")

        news_features[f"signal_{horizon}"] = (
            news_features[impact_key] * news_features["confidence"] * news_features["materiality"]
        )

        news_features[f"novelty_adjusted_signal_{horizon}"] = (
            news_features[f"signal_{horizon}"] * news_features["novelty"]
        )

        answer_features[f"signal_{horizon}"] = (
            answer_features[impact_key] * answer_features["confidence"]
        )

        answer_features[f"quality_adjusted_signal_{horizon}"] = (
            answer_features[f"signal_{horizon}"] * answer_features["information_quality"]
        )

        answer_features[f"novelty_adjusted_signal_{horizon}"] = (
            answer_features[f"signal_{horizon}"] * answer_features["novelty"])

    payload = {
        "date": str(feature_date.date()),
        "ticker": ticker,
        "company_name": (company_name),
        "category": (snapshot.get("category")),

        "model": MODEL_NAME,
        "base_url": BASE_URL,
        "prompt_version": (PROMPT_VERSION),
        "processor_version": (PROCESSOR_VERSION),
        "processed_at_utc": (datetime.now(timezone.utc).isoformat()),

        "news_features": (news_features),

        "answer_features": (answer_features),

        "raw_llm_responses": {
            "news": (news_raw_response),
            "answers": (answer_raw_response),
        },

        "input_files": {
            "snapshot_path": (str(snapshot_path)),
            "snapshot_sha256": (snapshot_hash),
            "answer_path": (str(answer_path) if answer_path is not None else None),
            "answer_sha256": (answer_hash),
            "previous_answer_path": (str(previous_answer_path) if previous_answer_path is not None else None),
            "previous_answer_sha256": (previous_answer_hash)
        },

        "input_processing": {
            "news_input_truncated": (news_input_truncated),
            "answer_input_truncated": (answer_input_truncated),
            "previous_answer_input_truncated": (previous_answer_input_truncated),
        }
    }

    write_json_atomically(output_path=output_path, payload=payload)

    return {
        "Date": feature_date,
        "Symbol": ticker,
        "Company": (company_name),
        "News_Available": (news_features["available"]),
        "News_Articles": (news_features["article_count"]),
        "Answer_Available": (answer_features["available"]),
        "Status": "processed",
        "Reason": reason,
        "Output_Path": (str(output_path))
    }

In [18]:
processing_records = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:

    future_to_path = {
        pool.submit(process_one_company_date, snapshot_path): snapshot_path

        for snapshot_path in snapshot_files
    }

    for future in tqdm(
        as_completed(future_to_path),
        total=len(future_to_path),
        desc=("Extracting LLM features")
    ):

        snapshot_path = (future_to_path[future])

        try:
            record = (future.result())

        except Exception as error:
            record = {
                "Date": (parse_date_from_folder(snapshot_path.parent)),
                "Symbol": None,
                "Company": None,
                "News_Available": None,
                "News_Articles": None,
                "Answer_Available": None,
                "Status": "failed",
                "Reason": ("processing_error"),
                "Output_Path": None,
                "Snapshot_Path": (str(snapshot_path)),
                "Error": (f"{type(error).__name__}: " f"{error}")
            }

        processing_records.append(record)


processing_log = pd.DataFrame(processing_records)

processing_log = (processing_log.sort_values(["Date", "Symbol"], na_position="last").reset_index(drop=True))

processing_log.to_csv(PROCESSING_LOG_PATH, index=False)

print(processing_log["Status"].value_counts(dropna=False))

display(processing_log.head(20))

Extracting LLM features:   0%|          | 0/4401 [00:00<?, ?it/s]

Status
processed    4243
skipped       140
failed         18
Name: count, dtype: int64


,Date,Symbol,Company,Status,Reason,Output_Path,News_Available,News_Articles,Answer_Available,Snapshot_Path,Error
0,2026-07-15,AAPL,Apple Inc.,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN
1,2026-07-15,ABNB,Airbnb,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN
2,2026-07-15,ACN,Accenture,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN
3,2026-07-15,ADBE,Adobe Inc.,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN
4,2026-07-15,AEP,American Electric Power,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN
5,2026-07-15,AES,AES Corporation,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN
6,2026-07-15,AFL,Aflac,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN
7,2026-07-15,AIG,American International Group,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN
8,2026-07-15,AMZN,Amazon,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN
9,2026-07-15,AOS,A. O. Smith,skipped,already_complete,/mnt/primary/Finnhub Pipeline/llm_financial_fe...,NaN,NaN,NaN,NaN,NaN


In [19]:
def flatten_feature_json(payload: dict[str, Any], source_path: Path) -> dict[str, Any]:

    news = (payload.get("news_features", {}) or {})

    answer = (payload.get("answer_features", {}) or {})

    input_files = (payload.get("input_files", {}) or {})

    input_processing = (payload.get("input_processing", {}) or {})

    return {
        "Date": pd.to_datetime(payload.get("date"), errors="coerce"),
        "Symbol": payload.get("ticker"),
        "Company": payload.get("company_name"),
        "Category": payload.get("category"),

        "News_Available": news.get("available"),
        "News_Sentiment": news.get("sentiment"),
        "News_Impact_1D": news.get("impact_1d"),
        "News_Impact_3D": news.get("impact_3d"),
        "News_Impact_5D": news.get("impact_5d"),
        "News_Confidence": news.get("confidence"),
        "News_Novelty": news.get("novelty"),
        "News_Materiality": news.get("materiality"),
        "News_Event_Type": news.get("event_type"),
        "News_Article_Count": news.get("article_count"),
        "News_Sentiment_Signal": news.get("sentiment_signal"),

        "News_Signal_1D": news.get("signal_1d"),
        "News_Signal_3D": news.get("signal_3d"),
        "News_Signal_5D": news.get("signal_5d"),

        "News_Novelty_Adjusted_Signal_1D": (news.get("novelty_adjusted_signal_1d")),
        "News_Novelty_Adjusted_Signal_3D": (news.get("novelty_adjusted_signal_3d")),
        "News_Novelty_Adjusted_Signal_5D": (news.get("novelty_adjusted_signal_5d")),

      
        "Answer_Available": answer.get("available"),
        "Answer_Impact_1D": answer.get("impact_1d"),
        "Answer_Impact_3D": answer.get("impact_3d"),
        "Answer_Impact_5D": answer.get("impact_5d"),
        "Answer_Confidence": answer.get("confidence"),
        "Answer_Fundamental_Strength": answer.get("fundamental_strength"),
        "Answer_Catalyst_Score": answer.get("catalyst_score"),
        "Answer_Risk_Score": answer.get("risk_score"),
        "Answer_Information_Quality": answer.get("information_quality"),
        "Answer_Novelty": answer.get("novelty"),
        "Answer_Previous_Available": answer.get("previous_available"),

        "Answer_Signal_1D": answer.get("signal_1d"),
        "Answer_Signal_3D": answer.get("signal_3d"),
        "Answer_Signal_5D": answer.get("signal_5d"),

        "Answer_Quality_Adjusted_Signal_1D": (answer.get("quality_adjusted_signal_1d")),
        "Answer_Quality_Adjusted_Signal_3D": (answer.get("quality_adjusted_signal_3d")),
        "Answer_Quality_Adjusted_Signal_5D": (answer.get("quality_adjusted_signal_5d")),

        "Answer_Novelty_Adjusted_Signal_1D": (answer.get("novelty_adjusted_signal_1d")),
        "Answer_Novelty_Adjusted_Signal_3D": (answer.get("novelty_adjusted_signal_3d")),
        "Answer_Novelty_Adjusted_Signal_5D": (answer.get("novelty_adjusted_signal_5d")),

     
        "Model": payload.get("model"),
        "Prompt_Version": payload.get("prompt_version"),
        "Processor_Version": payload.get("processor_version"),
        "Processed_At_UTC": payload.get("processed_at_utc"),
        "Snapshot_SHA256": (input_files.get("snapshot_sha256")),
        "Answer_SHA256": (input_files.get("answer_sha256")),
        "Snapshot_Path": (input_files.get("snapshot_path")),
        "Answer_Path": (input_files.get("answer_path")),
        "Previous_Answer_Path": (input_files.get("previous_answer_path")),
        "Previous_Answer_SHA256": (input_files.get("previous_answer_sha256")),
        "News_Input_Truncated": (input_processing.get("news_input_truncated")),
        "Answer_Input_Truncated": (input_processing.get("answer_input_truncated")),
        "Previous_Answer_Input_Truncated": (input_processing.get("previous_answer_input_truncated")),
        "Feature_JSON_Path": str(source_path)
    }


rows = []
invalid_files = []

for json_path in sorted(DAILY_JSON_ROOT.rglob("*_features.json")):

    try:
        with open(json_path, "r", encoding="utf-8") as file:
            payload = json.load(file)

        row = (flatten_feature_json( payload, json_path))

        if (pd.isna(row["Date"]) or not row["Symbol"]):
            raise ValueError("Missing Date or Symbol")

        rows.append(row)

    except Exception as error:
        invalid_files.append(
            {
                "Feature_JSON_Path": (str(json_path)),
                "Error": (f"{type(error).__name__}: " f"{error}"),
            }
        )


features_df = pd.DataFrame(rows)

if not features_df.empty:

    features_df["Date"] = (pd.to_datetime(features_df["Date"]).dt.normalize())

    features_df["Symbol"] = (features_df["Symbol"].astype(str).str.strip().str.upper())

    features_df = (features_df[(features_df["Date"] >= START_DATE) & (features_df["Date"] <= END_DATE)]
        .sort_values(["Date", "Symbol"])
        .drop_duplicates(["Date", "Symbol"], keep="last")
        .reset_index(drop=True)
    )

    features_df.to_csv(MASTER_CSV_PATH, index=False)


pd.DataFrame(invalid_files).to_csv(INVALID_JSON_REPORT_PATH, index=False)

print("Master rows:", len(features_df))

print("Companies:", (features_df["Symbol"].nunique() if not features_df.empty else 0))

if not features_df.empty:
    print("Range:", features_df["Date"].min().date(), "to", features_df["Date"].max().date())

print("Saved:", MASTER_CSV_PATH)

display(features_df.head())

Master rows: 4390
Companies: 100
Range: 2026-07-15 to 2026-08-28
Saved: /mnt/primary/Finnhub Pipeline/llm_financial_features/llm_financial_features.csv


,Date,Symbol,Company,Category,News_Available,News_Sentiment,News_Impact_1D,News_Impact_3D,News_Impact_5D,News_Confidence,...,Snapshot_SHA256,Answer_SHA256,Snapshot_Path,Answer_Path,Previous_Answer_Path,Previous_Answer_SHA256,News_Input_Truncated,Answer_Input_Truncated,Previous_Answer_Input_Truncated,Feature_JSON_Path
0,2026-07-15,AAPL,Apple Inc.,Information Technology,True,-0.20,-0.3,-0.20,0.0,0.70,...,0e3486059137089d5abbd9f7bc66f89a02d5f2f8db94b2...,8ed957aed9264419546baa3a62c2e6946a54cef50ec8d0...,/mnt/primary/Finnhub Pipeline/finnhub_snapshot...,/mnt/primary/Finnhub Pipeline/finnhub_answers/...,None,None,False,False,False,/mnt/primary/Finnhub Pipeline/llm_financial_fe...
1,2026-07-15,ABNB,Airbnb,Consumer Discretionary,True,0.35,0.2,0.25,0.2,0.78,...,50a231e91f7c2cbfa196511893796013b5707f3374bb51...,d72617050fef4ab54cb2d902f1f8937dae65c84993833d...,/mnt/primary/Finnhub Pipeline/finnhub_snapshot...,/mnt/primary/Finnhub Pipeline/finnhub_answers/...,None,None,False,False,False,/mnt/primary/Finnhub Pipeline/llm_financial_fe...
2,2026-07-15,ACN,Accenture,Information Technology,True,0.20,0.2,0.15,0.1,0.60,...,679210041121524a961158ea968330d2da0c4114810fb7...,59c5cb99bed062d320d2be8136321e6bc19045895bcafb...,/mnt/primary/Finnhub Pipeline/finnhub_snapshot...,/mnt/primary/Finnhub Pipeline/finnhub_answers/...,None,None,False,False,False,/mnt/primary/Finnhub Pipeline/llm_financial_fe...
3,2026-07-15,ADBE,Adobe Inc.,Information Technology,True,0.30,0.1,0.15,0.2,0.60,...,eb4d64005b9e0dacbb77efcc2124e23d4380f48c4d8c77...,9e1da45e448fc63c41d401a87f48076267f687a6d6234b...,/mnt/primary/Finnhub Pipeline/finnhub_snapshot...,/mnt/primary/Finnhub Pipeline/finnhub_answers/...,None,None,False,False,False,/mnt/primary/Finnhub Pipeline/llm_financial_fe...
4,2026-07-15,AEP,American Electric Power,Utilities,True,0.60,0.4,0.50,0.5,0.85,...,a29880815182d80d94806443976656cebf5ecfb2598ee9...,6586cefd6f5f7bd1463450014fb2d5ab42a0ffad713e35...,/mnt/primary/Finnhub Pipeline/finnhub_snapshot...,/mnt/primary/Finnhub Pipeline/finnhub_answers/...,None,None,False,False,False,/mnt/primary/Finnhub Pipeline/llm_financial_fe...


In [20]:
SIGNED_COLUMNS = [
    "News_Sentiment",
    "News_Impact_1D",
    "News_Impact_3D",
    "News_Impact_5D",
    "Answer_Impact_1D",
    "Answer_Impact_3D",
    "Answer_Impact_5D",
    "Answer_Fundamental_Strength",
    "Answer_Catalyst_Score",
]

UNSIGNED_COLUMNS = [
    "News_Confidence",
    "News_Novelty",
    "News_Materiality",
    "Answer_Confidence",
    "Answer_Risk_Score",
    "Answer_Information_Quality",
    "Answer_Novelty",
]


for column in SIGNED_COLUMNS:
    valid = (features_df[column].dropna().between(-1, 1).all())

    print(f"{column}: [-1,1] ->", valid)


for column in UNSIGNED_COLUMNS:
    valid = (features_df[column].dropna().between(0, 1).all())
    print(f"{column}: [0,1] ->", valid)

News_Sentiment: [-1,1] -> True
News_Impact_1D: [-1,1] -> True
News_Impact_3D: [-1,1] -> True
News_Impact_5D: [-1,1] -> True
Answer_Impact_1D: [-1,1] -> True
Answer_Impact_3D: [-1,1] -> True
Answer_Impact_5D: [-1,1] -> True
Answer_Fundamental_Strength: [-1,1] -> True
Answer_Catalyst_Score: [-1,1] -> True
News_Confidence: [0,1] -> True
News_Novelty: [0,1] -> True
News_Materiality: [0,1] -> True
Answer_Confidence: [0,1] -> True
Answer_Risk_Score: [0,1] -> True
Answer_Information_Quality: [0,1] -> True
Answer_Novelty: [0,1] -> True


In [21]:
if features_df.empty:
    coverage_report = (pd.DataFrame())

else:

    coverage_report = (features_df.groupby("Date")
        .agg(Companies=("Symbol", "nunique"),
            Companies_With_News=("News_Available", lambda values: int(values.fillna(False).sum())),
            Companies_With_Answers=("Answer_Available", lambda values: int(values.fillna(False).sum())),
            Companies_With_Previous_Answers=("Answer_Previous_Available", lambda values: int(values.fillna(False).sum())),
            Total_Filtered_News_Articles=("News_Article_Count", "sum"),
            News_Inputs_Truncated=("News_Input_Truncated", lambda values: int(values.fillna(False).sum())),
            Answer_Inputs_Truncated=("Answer_Input_Truncated", lambda values: int(values.fillna(False).sum())))
        .reset_index()
        .sort_values("Date")
    )


coverage_report.to_csv(COVERAGE_REPORT_PATH, index=False)

display(coverage_report)

,Date,Companies,Companies_With_News,Companies_With_Answers,Companies_With_Previous_Answers,Total_Filtered_News_Articles,News_Inputs_Truncated,Answer_Inputs_Truncated
0,2026-07-15,100,96,100,0,613,0,0
1,2026-07-16,100,97,100,100,629,0,0
2,2026-07-17,100,97,100,100,588,0,0
3,2026-07-18,100,96,100,100,576,0,0
4,2026-07-19,100,96,100,100,563,0,0
5,2026-07-20,100,96,100,100,565,0,0
6,2026-07-21,100,95,100,100,566,0,0
7,2026-07-22,100,97,100,100,586,0,0
8,2026-07-23,100,97,100,100,621,0,0
9,2026-07-24,100,96,100,100,594,0,0


In [22]:
TICKER_TO_INSPECT = "AAPL"

display_columns = [
    "Date",
    "Symbol",

    "News_Available",
    "News_Sentiment",
    "News_Impact_1D",
    "News_Impact_3D",
    "News_Impact_5D",
    "News_Confidence",
    "News_Novelty",
    "News_Materiality",
    "News_Event_Type",
    "News_Signal_1D",
    "News_Signal_3D",
    "News_Signal_5D",

    "Answer_Available",
    "Answer_Impact_1D",
    "Answer_Impact_3D",
    "Answer_Impact_5D",
    "Answer_Confidence",
    "Answer_Fundamental_Strength",
    "Answer_Catalyst_Score",
    "Answer_Risk_Score",
    "Answer_Information_Quality",
    "Answer_Novelty",
    "Answer_Previous_Available",
    "Answer_Signal_1D",
    "Answer_Signal_3D",
    "Answer_Signal_5D",
    "Answer_Novelty_Adjusted_Signal_1D",
    "Answer_Novelty_Adjusted_Signal_3D",
    "Answer_Novelty_Adjusted_Signal_5D"
]

company_features = (features_df[features_df["Symbol"] == TICKER_TO_INSPECT].copy())

display(company_features[display_columns])

,Date,Symbol,News_Available,News_Sentiment,News_Impact_1D,News_Impact_3D,News_Impact_5D,News_Confidence,News_Novelty,News_Materiality,...,Answer_Risk_Score,Answer_Information_Quality,Answer_Novelty,Answer_Previous_Available,Answer_Signal_1D,Answer_Signal_3D,Answer_Signal_5D,Answer_Novelty_Adjusted_Signal_1D,Answer_Novelty_Adjusted_Signal_3D,Answer_Novelty_Adjusted_Signal_5D
0,2026-07-15,AAPL,True,-0.20,-0.30,-0.20,0.00,0.70,0.50,0.80,...,0.30,0.70,0.00,False,0.1100,0.1650,0.165,0.00000,0.000000,0.00000
100,2026-07-16,AAPL,True,0.60,0.70,0.50,0.40,0.65,0.70,0.85,...,0.30,0.70,0.80,True,0.0600,0.0900,0.120,0.04800,0.072000,0.09600
200,2026-07-17,AAPL,True,0.30,0.20,0.15,0.10,0.60,0.70,0.80,...,0.40,0.60,0.90,True,0.0500,0.1000,0.100,0.04500,0.090000,0.09000
300,2026-07-18,AAPL,True,0.30,0.20,0.15,0.10,0.60,0.40,0.50,...,0.30,0.70,0.50,True,0.0600,0.1200,0.120,0.03000,0.060000,0.06000
400,2026-07-19,AAPL,True,0.30,0.20,0.15,0.10,0.65,0.70,0.80,...,0.40,0.70,0.70,True,0.1000,0.0750,0.050,0.07000,0.052500,0.03500
500,2026-07-20,AAPL,True,0.30,0.20,0.25,0.30,0.55,0.70,0.65,...,0.50,0.60,0.70,True,0.0350,0.0525,0.070,0.02450,0.036750,0.04900
600,2026-07-21,AAPL,True,-0.10,-0.10,-0.05,0.00,0.40,0.30,0.60,...,0.60,0.70,0.90,True,0.0000,0.0000,0.000,0.00000,0.000000,0.00000
700,2026-07-22,AAPL,True,-0.10,0.00,-0.10,-0.10,0.60,0.80,0.60,...,0.30,0.85,0.70,True,0.0000,0.0000,0.000,0.00000,0.000000,0.00000
800,2026-07-23,AAPL,True,0.00,-0.10,-0.05,0.00,0.50,0.70,0.60,...,0.40,0.60,0.70,True,0.0300,0.0450,0.060,0.02100,0.031500,0.04200
900,2026-07-24,AAPL,True,-0.10,-0.05,-0.10,-0.10,0.55,0.60,0.60,...,0.30,0.60,0.90,True,0.0000,0.0000,0.000,0.00000,0.000000,0.00000


In [23]:
feature_columns = [
    "News_Sentiment",
    "News_Impact_1D",
    "News_Impact_3D",
    "News_Impact_5D",
    "News_Confidence",
    "News_Novelty",
    "News_Materiality",
    "News_Signal_1D",
    "News_Signal_3D",
    "News_Signal_5D",

    "Answer_Impact_1D",
    "Answer_Impact_3D",
    "Answer_Impact_5D",
    "Answer_Confidence",
    "Answer_Fundamental_Strength",
    "Answer_Catalyst_Score",
    "Answer_Risk_Score",
    "Answer_Information_Quality",
    "Answer_Novelty",
    "Answer_Signal_1D",
    "Answer_Signal_3D",
    "Answer_Signal_5D",
    "Answer_Novelty_Adjusted_Signal_1D",
    "Answer_Novelty_Adjusted_Signal_3D",
    "Answer_Novelty_Adjusted_Signal_5D"
]

display(features_df[feature_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
News_Sentiment,4390.0,0.153572,0.330355,-0.90000,-0.10000,0.200000,0.400000,0.9000
News_Impact_1D,4390.0,0.081235,0.264084,-0.90000,-0.10000,0.100000,0.200000,0.9000
News_Impact_3D,4390.0,0.103834,0.253017,-0.70000,-0.05000,0.100000,0.250000,0.8000
News_Impact_5D,4390.0,0.097298,0.217996,-0.60000,0.00000,0.100000,0.200000,0.8000
News_Confidence,4390.0,0.594638,0.199297,0.00000,0.55000,0.600000,0.700000,0.9500
News_Novelty,4390.0,0.535861,0.245605,0.00000,0.40000,0.600000,0.700000,0.9200
News_Materiality,4390.0,0.619651,0.242797,0.00000,0.50000,0.700000,0.800000,1.0000
News_Signal_1D,4390.0,0.046306,0.156048,-0.72900,-0.02400,0.030000,0.102000,0.6885
News_Signal_3D,4390.0,0.056525,0.147635,-0.54150,-0.00945,0.031688,0.119000,0.6120
News_Signal_5D,4390.0,0.052431,0.127710,-0.45125,0.00000,0.024000,0.099000,0.6120
